In [0]:
# Step 1: Import the libraries needed for file ingestion
# and merging new records into an existing Delta table.

from pyspark.sql import functions as F
from delta.tables import DeltaTable


# Get the catalog currently being used in Databricks.
catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_name = "retail_lakehouse"

# Define the location where the retail CSV files are stored.
landing_path = (
    f"/Volumes/{catalog}/{schema_name}/retail_landing"
)

# Create a separate folder for the next incoming batch.
incoming_path = f"{landing_path}/incoming_orders_batch_02"


# Step 2: Prepare sample records to simulate new orders.
# O2010 appears twice so we can test duplicate handling.

batch_data = [
    ("O2010", "2026-10-10", "C010", "P101",
     2, 499.0, "Completed", "West"),

    ("O2011", "2026-10-10", "C011", "P102",
     1, 999.0, "Completed", "South"),

    ("O2010", "2026-10-10", "C010", "P101",
     2, 499.0, "Completed", "West")
]

columns = [
    "order_id", "order_date", "customer_id", "product_id",
    "quantity", "unit_price", "status", "region"
]

# Convert the sample records into a Spark DataFrame.
batch_df = spark.createDataFrame(batch_data, columns)

display(batch_df)

In [0]:
# Step 3: Save the incoming records as CSV files.
# This simulates receiving a new batch from an upstream system.

(
    batch_df.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(incoming_path)
)

# Print the folder location so we can reuse it in the next cell.
print("Incoming batch saved to:", incoming_path)

In [0]:
# Step 4: Read the saved CSV files back into Spark.
# Use the expected schema and convert the order date to a date type.

incoming_df = (
    spark.read
    .option("header", True)
    .schema(batch_df.schema)
    .csv(incoming_path)
    .withColumn(
        "order_date",
        F.to_date("order_date", "yyyy-MM-dd")
    )
)

# Check that all three sample records were read.
print("Incoming rows:", incoming_df.count())

display(incoming_df)

In [0]:
# Step 5: Remove repeated order IDs within the incoming batch.
# For this sample, order_id is treated as the unique business key.

batch_deduplicated = incoming_df.dropDuplicates(
    ["order_id"]
)

# Define basic rules for accepting a record.
# Invalid quantities and negative prices are not allowed.

valid_condition = (
    F.col("order_id").isNotNull() &
    F.col("order_date").isNotNull() &
    F.col("product_id").isNotNull() &
    F.col("quantity").isNotNull() &
    (F.col("quantity") > 0) &
    F.col("unit_price").isNotNull() &
    (F.col("unit_price") >= 0)
)

# Keep valid records for ingestion.
valid_batch = batch_deduplicated.filter(
    valid_condition
)

# Keep invalid records separate for inspection.
rejected_batch = batch_deduplicated.filter(
    ~valid_condition
)

print("Rows after deduplication:", batch_deduplicated.count())
print("Valid rows:", valid_batch.count())
print("Rejected rows:", rejected_batch.count())

display(valid_batch)
display(rejected_batch)

In [0]:
# Step 6: Identify the existing Bronze orders table.
# The table must already exist and have matching columns.

bronze_orders_table = (
    f"{catalog}.{schema_name}.bronze_orders"
)

bronze_delta = DeltaTable.forName(
    spark,
    bronze_orders_table
)

# Insert orders that do not already exist.
# Existing order IDs are left unchanged, preventing
# repeated deliveries from inserting the same order again.

(
    bronze_delta.alias("target")
    .merge(
        valid_batch.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Bronze merge completed.")

In [0]:
# Step 7: Read the updated Bronze table.
# Confirm that both incoming orders are present.

bronze_orders = spark.table(
    bronze_orders_table
)

new_order_count = bronze_orders.filter(
    F.col("order_id").isin("O2010", "O2011")
).count()

# Check whether any order IDs appear more than once.
duplicate_ids = (
    bronze_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Incoming orders present:", new_order_count)
print("Duplicate order IDs:", duplicate_ids)

# Stop the notebook if the validation conditions fail.
assert new_order_count == 2, (
    "Expected both new orders to be present."
)

assert duplicate_ids == 0, (
    "Duplicate order IDs were found in Bronze."
)

print("Incremental ingestion validation passed.")

In [0]:
# Step 8: Inspect the newly ingested orders.
# This makes it easier to verify the merge result manually.

display(
    bronze_orders
    .filter(
        F.col("order_id").isin("O2010", "O2011")
    )
    .orderBy("order_id")
)